In [34]:
train_path = r"F:\Uni a11\fourth\sec\123\DeepX_train.xlsx"
val_path = r"F:\Uni a11\fourth\sec\123\DeepX_validation.xlsx"
unlabeled_path = r"F:\Uni a11\fourth\sec\123\DeepX_hidden_test.xlsx"

In [35]:
import pandas as pd
import numpy as np

train = pd.read_excel(train_path)
test = pd.read_excel(val_path)
unlab = pd.read_excel(unlabeled_path)

In [37]:
train.isna().sum()

review_id            0
review_text          0
star_rating          0
date                 0
business_name        0
business_category    0
platform             0
aspects              0
aspect_sentiments    0
dtype: int64

In [38]:
test.isna().sum()

review_id            0
review_text          0
star_rating          0
date                 0
business_name        0
business_category    0
platform             0
aspects              0
aspect_sentiments    0
dtype: int64

In [39]:
unlab.isna().sum()

review_id            0
review_text          0
star_rating          0
date                 0
business_name        0
business_category    0
platform             0
dtype: int64

In [40]:
import pandas as pd
import re

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002700-\U000027BF"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE
)

def has_emoji(text):
    return bool(emoji_pattern.search(str(text)))

In [53]:
def has_franco(text):
    text = str(text).lower()
    
    return bool(
        re.search(r"[a-zA-Z]", text) and re.search(r"[3-9]", text)
    )

In [54]:
train["has_emoji"] = train["review_text"].apply(has_emoji)
train["has_franco"] = train["review_text"].apply(has_franco)

In [55]:
test["has_emoji"] = test["review_text"].apply(has_emoji)
test["has_franco"] = test["review_text"].apply(has_franco)

In [56]:
unlab["has_emoji"] = unlab["review_text"].apply(has_emoji)
unlab["has_franco"] = unlab["review_text"].apply(has_franco)

In [57]:
def analyze_df(df, name, text_col="review_text"):
    print(f"\n===== {name} =====")

    # 1. duplicates
    dup_count = df.duplicated().sum()
    print("Duplicates:", dup_count)

    # 2. emoji rows
    emoji_count = df[text_col].apply(has_emoji).sum()
    print("Rows with emojis:", emoji_count)

    # 3. franco rows
    franco_count = df[text_col].apply(has_franco).sum()
    print("Rows with franco:", franco_count)

    # optional: percentages
    total = len(df)
    print("Total rows:", total)
    print("Emoji %:", round(emoji_count / total * 100, 2))
    print("Franco %:", round(franco_count / total * 100, 2))

In [58]:
train = pd.read_excel(train_path)
test = pd.read_excel(val_path)
unlab = pd.read_excel(unlabeled_path)

analyze_df(train, "TRAIN")
analyze_df(test, "TEST")
analyze_df(unlab, "UNLABELED")


===== TRAIN =====
Duplicates: 0
Rows with emojis: 191
Rows with franco: 13
Total rows: 1971
Emoji %: 9.69
Franco %: 0.66

===== TEST =====
Duplicates: 0
Rows with emojis: 50
Rows with franco: 1
Total rows: 500
Emoji %: 10.0
Franco %: 0.2

===== UNLABELED =====
Duplicates: 0
Rows with emojis: 59
Rows with franco: 2
Total rows: 500
Emoji %: 11.8
Franco %: 0.4


In [59]:
print("TRAIN duplicates:", train.duplicated().sum())
print("TEST duplicates:", test.duplicated().sum())
print("UNLABELED duplicates:", unlab.duplicated().sum())

TRAIN duplicates: 0
TEST duplicates: 0
UNLABELED duplicates: 0


In [63]:
all_data = pd.concat([train, test, unlab], ignore_index=True)

print("Global duplicates (full row):", all_data.duplicated().sum())

Global duplicates (full row): 0


In [65]:
test["text_len"] = test["review_text"].astype(str).apply(len)

test["text_len"].describe()

count     500.000000
mean      109.796000
std       125.641258
min         2.000000
25%        36.000000
50%        77.000000
75%       140.250000
max      1159.000000
Name: text_len, dtype: float64

In [66]:
train["text_len"] = train["review_text"].astype(str).apply(len)

train["text_len"].describe()

count    1971.000000
mean      119.223237
std       145.344526
min         2.000000
25%        36.000000
50%        75.000000
75%       148.000000
max      1526.000000
Name: text_len, dtype: float64

In [67]:
def is_arabic(text):
    return bool(re.search(r"[\u0600-\u06FF]", str(text)))

def is_english(text):
    return bool(re.search(r"[a-zA-Z]", str(text)))

In [68]:
repeated_pattern = re.compile(r"(.)\1{3,}")

def has_repeated_chars(text):
    return bool(repeated_pattern.search(str(text)))

In [69]:
def is_noise(text):
    text = str(text).strip()
    return len(re.sub(r"[\W_]+", "", text)) == 0

In [70]:
has_url = lambda x: bool(re.search(r"http\S+|www\.", str(x)))
has_email = lambda x: bool(re.search(r"\S+@\S+", str(x)))

In [29]:
#!/usr/bin/env python3
"""
=======================================================================
  ARABIC ASPECT-BASED SENTIMENT ANALYSIS (ABSA)  — DeepX Challenge
  Full Production Pipeline
=======================================================================

HOW TO RUN:
  # With real data files:
  python3 arabic_absa_full.py --train DeepX_train.xlsx \
                               --val   DeepX_validation.xlsx \
                               --unlabeled DeepX_unlabeled.xlsx \
                               --output predictions.json

  # Demo mode (no files needed):
  python3 arabic_absa_full.py --demo

REQUIREMENTS:
  pip install pandas openpyxl scikit-learn emoji langdetect

OUTPUT:
  predictions.json  — Array of {review_id, aspects, aspect_sentiments}
=======================================================================
"""

import re, json, sys, os, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.calibration import CalibratedClassifierCV



ALEF_NORM   = str.maketrans("أإآٱ", "اااا")
WAW_NORM    = str.maketrans("ؤ",    "و")
YEH_NORM    = str.maketrans("ئىي", "ييي")
TA_MARBUTA  = str.maketrans("ة",    "ه")
KASHIDA     = "\u0640"
DIACRITICS  = re.compile(r"[\u064b-\u065f\u0670]")
ZERO_WIDTH  = re.compile(r"[\u200b-\u200f\u202a-\u202e\ufeff]")
ARABIC_PUNC = re.compile(r"[،؟؛«»\u064b-\u065f]")
PUNCTUATION = re.compile(r"[^\w\s\u0600-\u06ff]", re.UNICODE)
NUMBERS     = re.compile(r"\d+")
EXTRA_WS    = re.compile(r"\s+")
TOKEN_PAT   = re.compile(r"[\w\u0600-\u06ff]+", re.UNICODE)
REPEATED    = re.compile(r"(.)\1{2,}")


def normalize_arabic(text: str) -> str:
    text = text.translate(ALEF_NORM).translate(WAW_NORM)
    text = text.translate(YEH_NORM).translate(TA_MARBUTA)
    text = text.replace(KASHIDA, "")
    text = DIACRITICS.sub("", text)
    text = ZERO_WIDTH.sub("", text)
    return text


POSITIVE_EMOJIS = set("❤️😍🥰😊😁😄😃🎉🔥💪👍✅💯🌹🌺🏆⭐🌟💫✨🥳😋🤩")
NEGATIVE_EMOJIS = set("😠😡🤬😤😒😞😔💔👎❌🚫😢😭🤢🤮😫😩")
EMOJI_RE = re.compile(
    "["
    "\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F9FF\u2600-\u26FF\u2700-\u27BF"
    "]+", flags=re.UNICODE)


def handle_emojis(text: str) -> Tuple[str, str]:
    sentiments = []
    def replacer(m):
        for ch in m.group(0):
            sentiments.append("positive" if ch in POSITIVE_EMOJIS
                              else "negative" if ch in NEGATIVE_EMOJIS
                              else "neutral")
        return " _EMOJI_ "
    cleaned = EMOJI_RE.sub(replacer, text)
    if not sentiments:
        return cleaned, "neutral"
    pos = sentiments.count("positive")
    neg = sentiments.count("negative")
    hint = "positive" if pos > neg else ("negative" if neg > pos else "neutral")
    return cleaned, hint


FRANCO_DICT: Dict[str, str] = {
    "ana":"انا","enta":"انت","enti":"انت","howa":"هو","heya":"هي",
    "ehna":"احنا","msh":"مش","mesh":"مش","m4":"مش","bs":"بس","bas":"بس",
    "3shan":"عشان","3la":"على","3al":"على","b3d":"بعد","kda":"كده",
    "wla":"ولا","walla":"والله","wallah":"والله","tayeb":"طيب",
    "tamam":"تمام","ok":"تمام","okay":"تمام","meen":"مين","feen":"فين",
    "emta":"امتى","leh":"ليه","eih":"ايه","ayh":"ايه","ya3ni":"يعني",
    "tab3an":"طبعا","tab3":"طبعا","momken":"ممكن","momkin":"ممكن",
    "kwayyes":"كويس","kwayes":"كويس","gameel":"جميل","gamed":"جامد",
    "mafish":"مافيش","lazem":"لازم","3andak":"عندك","3andi":"عندي",
    "mashy":"ماشي","mashi":"ماشي","ahlan":"اهلا","shukran":"شكرا",
    "merci":"شكرا","afwan":"عفوا","inshallah":"انشاء الله",
    "insha2allah":"انشاء الله","habibi":"حبيبي","habibti":"حبيبتي",
    "yalla":"يلا","yala":"يلا","khalas":"خلاص","5alas":"خلاص",
    "3ayez":"عايز","3ayza":"عايزه","7elwa":"حلوه","7elo":"حلو",
    "7elw":"حلو","sa7":"صح","sa7i7":"صحيح","7aram":"حرام",
    "7alal":"حلال","tayb":"طيب","zain":"زين","mnee7":"منيح",
    "b3den":"بعدين","delwati":"دلوقتي","dilwaqti":"دلوقتي",
}

ARABIZI_MAP = [
    ("sh","ش"),("gh","غ"),("kh","خ"),("th","ث"),("dh","ذ"),("zh","ز"),
    ("ch","ش"),("ph","ف"),("2","ء"),("3","ع"),("5","خ"),("6","ط"),
    ("7","ح"),("8","غ"),("9","ق"),("a","ا"),("b","ب"),("t","ت"),
    ("g","ج"),("d","د"),("r","ر"),("z","ز"),("s","س"),("f","ف"),
    ("k","ك"),("l","ل"),("m","م"),("n","ن"),("h","ه"),("w","و"),
    ("y","ي"),("e","ي"),("o","و"),("u","و"),("i","ي"),("p","ب"),
    ("v","ف"),("j","ج"),("q","ق"),("x","ش"),
]


def _is_arabizi(tok: str) -> bool:
    return bool(re.search(r"[a-zA-Z]", tok)) and bool(re.search(r"[2356789]", tok))


def _xliterate(tok: str) -> str:
    t, i = "", 0
    tok_l = tok.lower()
    while i < len(tok_l):
        matched = False
        for lat, ar in ARABIZI_MAP:
            if tok_l[i:i+len(lat)] == lat:
                t += ar; i += len(lat); matched = True; break
        if not matched:
            t += tok_l[i]; i += 1
    return t


def handle_franco(text: str) -> str:
    tokens = text.split()
    out = []
    for tok in tokens:
        clean = tok.lower().strip("!?.,;:")
        if clean in FRANCO_DICT:
            out.append(FRANCO_DICT[clean])
        elif _is_arabizi(tok):
            out.append(_xliterate(tok))
        else:
            out.append(tok)
    return " ".join(out)


STOPWORDS = {
    "في","من","إلى","الى","على","عن","مع","هذا","هذه","ذلك","تلك",
    "هو","هي","هم","هن","انا","نحن","انت","انتم","كان","كانت","كانوا",
    "يكون","تكون","لا","لم","لن","ما","ان","إن","أن","لكن","لأن","لان",
    "لذا","لذلك","إذا","اذا","عند","عندما","حين","حتى","قد","قبل","بعد",
    "كل","بعض","أي","اي","به","بها","بهم","له","لها","لهم","منه","منها",
    "منهم","فيه","فيها","عليه","عليها","عنه","وهو","وهي","وهم","التي",
    "الذي","الذين","اللذين","اللواتي","ايضا","أيضا","جدا","جداً","اكثر",
    "أكثر","اقل","اول","اخر","كما","مما","بما","إذ","اذ","بل","أو","او",
    "ثم","و","ف","ب","ل","ك","أ","إ","فقط",
}


AR_PFXS = sorted(
    ["ال","وال","بال","فال","وب","لل","فب","وف","ول","و","ف","ب","ل","ك"],
    key=len, reverse=True)
AR_SFXS = sorted(
    ["تين","ين","ون","ان","ات","ها","هم","هن","كم","كن","نا","وا","تا",
     "ية","يه","ه","ا","ة","ي","ك"],
    key=len, reverse=True)


def stem_word(w: str) -> str:
    for p in AR_PFXS:
        if w.startswith(p) and len(w)-len(p) >= 3:
            w = w[len(p):]; break
    for s in AR_SFXS:
        if w.endswith(s) and len(w)-len(s) >= 3:
            w = w[:-len(s)]; break
    return w



def preprocess(text: str, stem: bool = True, remove_stops: bool = True,
               return_steps: bool = False) -> dict:
    steps = {}
    text = text.strip()
    steps["original"] = text

    text, emoji_hint = handle_emojis(text)
    steps["after_emoji"] = text

    text = handle_franco(text)
    steps["after_franco"] = text

    text = normalize_arabic(text)
    steps["after_normalization"] = text

    text = REPEATED.sub(r"\1\1", text)
    steps["after_noise_removal"] = text

    text = ARABIC_PUNC.sub(" ", text)
    text = PUNCTUATION.sub(" ", text)
    steps["after_punctuation"] = text

    text = NUMBERS.sub(" ", text)
    steps["after_numbers"] = text

    text = EXTRA_WS.sub(" ", text).strip()
    steps["after_whitespace"] = text

    tokens = TOKEN_PAT.findall(text)
    steps["tokens"] = tokens

    if remove_stops:
        tokens = [t for t in tokens if t.lower() not in STOPWORDS and len(t) > 1]
    steps["after_stopwords"] = tokens

    if stem:
        tokens = [stem_word(t) for t in tokens]
    steps["after_stemming"] = tokens

    return {
        "clean_text": " ".join(tokens),
        "tokens": tokens,
        "emoji_hint": emoji_hint,
        **({"steps": steps} if return_steps else {}),
    }




ASPECTS = ["food","service","price","cleanliness","delivery",
           "ambiance","app_experience","general","none"]

ASPECT_KEYWORDS: Dict[str, List[str]] = {
    "food": [
        "اكل","طعام","وجبه","وجبات","مطبخ","طبخ","طبق","اطباق","مذاق",
        "نكهه","طعم","لذيذ","لذه","حلو","مالح","حار","ساخن","بارد","فاسد",
        "طازج","ناضج","نيء","برجر","بيتزا","شاورما","كباب","مشويات","سوشي",
        "مكرونه","سلطه","شوربه","حساء","كنافه","حلويات","كيك","قهوه",
        "عصير","مشروب","شاي","ايسكريم","بوفيه","منيو","قائمه","فطار",
        "غداء","عشاء","حجم","كميه","بروتين","خضار","خضروات","فاكهه",
        "مخبز","عجين","خبز","معجنات","بايخ","رائحه",
    ],
    "service": [
        "خدمه","موظف","موظفين","طاقم","نادل","نادله","كاشير","مدير",
        "استقبال","تعامل","اهتمام","مساعده","سرعه","بطء","انتظار","وقت",
        "سلوك","احترام","ادب","وقاحه","ابتسامه","ترحيب","خدمه العملاء",
        "ردود","تجاوب","محترم","متاخر","بطيء","حجز","طلب","اوردر",
        "طلبات","رد","شكوى","مشكله","حل","متعاون","ودود","كفاءه",
    ],
    "price": [
        "سعر","اسعار","تكلفه","فلوس","غالي","رخيص","معقول","مبالغ","ثمن",
        "دفع","مجاني","مجانا","خصم","عرض","عروض","قيمه","ميزانيه",
        "فاتوره","حساب","تمن","كاش","بطاقه","ماستركارد","فيزا","بريميوم",
        "اشتراك","رسوم","مش يستاهل","مش مستاهل",
    ],
    "cleanliness": [
        "نضافه","نظافه","نضيف","نظيف","وسخ","قذر","اتساخ","قذاره",
        "حمام","دورات مياه","مياه","بيئه","صحه","صحي","غير صحي",
        "تعقيم","تنظيف","ريحه","روائح","حشرات","صراصير","فئران",
        "اتربه","غبار",
    ],
    "delivery": [
        "توصيل","توصيله","ديليفري","سرعه التوصيل","بطء التوصيل","تاخير",
        "تاخر","وصل","وصول","شحن","تتبع","مندوب","مندوبين","سائق",
        "طرد","تغليف","تعبئه","استلام","ما وصل","لم يصل","رفض التوصيل",
        "مسافه","منطقه","نطاق التوصيل",
    ],
    "ambiance": [
        "اجواء","جو","ديكور","تصميم","اضاءه","اناره","موسيقى","موسيقي",
        "ضوضاء","هدوء","هادي","صاخب","مكان","جلسه","قعده","كراسي",
        "طاولات","مقاعد","واسع","ضيق","مريح","غير مريح","رومانسي",
        "عائلي","ترفيه","منظر","اطلاله","حديقه","جنينه","داخلي","خارجي",
        "شاليه","استراحه",
    ],
    "app_experience": [
        "تطبيق","ابليكيشن","برنامج","موقع","سايت","واجهه","سهوله",
        "صعوبه","بطء التطبيق","سرعه التطبيق","تحديث","بلاي ستور",
        "ابل ستور","اندرويد","ايفون","تسجيل","دخول","باسورد","بحث",
        "فلتر","ميزه","خاصيه","ازرار","خريطه","دفع الكتروني","ادفع",
        "بطاقه","حذف","خطا","مشكله تقنيه","انهيار","كراش","تعليق",
        "فريز","بطيء","سريع","سهل الاستخدام","معقد","مفيد","حمل",
        "تنزيل","تحميل","اشتراك",
    ],
    "general": [
        "ممتاز","رائع","تحفه","جميل","احسن","افضل","سيء","مخيب",
        "محتاج تطوير","ينصح","اوصي","راجع","ارجع","زياره","تجربه",
        "تجربتي","اول مره","دايم","مستمر","بالتوفيق","ماشاء الله",
        "الحمد لله","اوكي","عادي","كويس","ما عجبني","كل شيء",
        "عموما","بشكل عام","ممتاز جداً","رائع جداً",
    ],
}

ASPECT_TOKEN_MAP: Dict[str, str] = {}
for asp, kws in ASPECT_KEYWORDS.items():
    for kw in kws:
        ASPECT_TOKEN_MAP[kw] = asp

POS_LEXICON = {
    "ممتاز","رائع","تحفه","جميل","احسن","لذيذ","لذيذه","ناجح","مميز",
    "مثالي","فوق الممتاز","محترم","نضيف","نظيف","سريع","دقيق","مفيد",
    "ودود","مريح","مبهج","جذاب","فخم","منظم","احترافي","منيح","عظيم",
    "بديع","رائعه","جميله","حلو","حلوه","كويس","كويسه","تمام","صح",
    "موفق","اعجبني","شكرا","احب","افضل","الافضل","ينصح","اوصي",
    "يستاهل","مستاهل","قيمه","زين","بخير","طيب","سهل","واضح","خدمه ممتازه",
    "جوده عاليه","نظيف جدا","لذيذ جدا","رائع جدا","ممتاز جدا",
    "حلو جدا","جميل جدا","مره حلو","تمام جدا","مفيد جدا",
}

NEG_LEXICON = {
    "سيء","وحش","مش كويس","مش حلو","مكسور","غالي","محتال","بطيء",
    "متاخر","قذر","وسخ","بارد","فاسد","مش طازج","سيئه","رديء","رديئه",
    "مخيب","محبط","مزعج","تعبان","ما عجبني","ما اشتريت","لن اعود",
    "ما ارجع","مريض","كارثه","مصيبه","مشكله","اشتكي","شاكي","شكوى",
    "حرام","سرقه","نصابه","احتياله","ضعيف","ضعيفه","منتهي صلاحيه",
    "قديم","بائخ","بايخ","ثقيل","تعب","مش منطقي","ما يستاهل",
    "مبالغ فيه","مش مستوى","خدمه سيئه","موظف وقح","انتظار طويل",
    "مش مريح","غير مريح","غير نظيف","متسخ","مزعج جدا","سيء جدا",
}

NEGATIONS = {
    "لا","لم","لن","ما","مش","مو","غير","ليس","ليست","لايوجد",
    "لا يوجد","ماعندي","مافيه","ماكو","مو","بدون",
}

INTENSIFIERS = {
    "جدا","جداً","كثير","اوي","اووي","جامد","فوق","زياده",
    "مره","الله","ماشاء الله",
}

WINDOW = 4


def _score_window(tokens: List[str], center: int) -> float:
    start = max(0, center - WINDOW)
    end   = min(len(tokens), center + WINDOW + 1)
    score, negated = 0.0, False
    for tok in tokens[start:end]:
        if tok in NEGATIONS:    negated = True
        elif tok in INTENSIFIERS: score *= 1.4
        elif tok in POS_LEXICON:  score += 1.0
        elif tok in NEG_LEXICON:  score -= 1.0
    if negated: score = -score
    return max(-1.0, min(1.0, score))


def _score_to_label(score: float, threshold: float = 0.12) -> str:
    if score > threshold:  return "positive"
    if score < -threshold: return "negative"
    return "neutral"




def rule_detect_aspects(raw_tokens: List[str], stem_tokens: List[str],
                        platform: str = "") -> List[Tuple[str, int]]:
    found: Dict[str, int] = {}
    for idx, (raw, stem) in enumerate(zip(raw_tokens, stem_tokens)):
        for tok in [raw.lower(), stem.lower()]:
            if tok in ASPECT_TOKEN_MAP:
                asp = ASPECT_TOKEN_MAP[tok]
                if asp not in found:
                    found[asp] = idx
            for kw, asp in ASPECT_TOKEN_MAP.items():
                if len(kw) > 5 and kw in tok and asp not in found:
                    found[asp] = idx

    if platform in ("play_store","app_store") and "app_experience" not in found:
        found["app_experience"] = 0

    if not found:
        found["general"] = 0

    return list(found.items())


def rule_classify_sentiment(stem_tokens: List[str],
                             aspect_positions: List[Tuple[str, int]],
                             star_rating: Optional[float] = None,
                             emoji_hint: str = "neutral") -> Dict[str, str]:
    if star_rating is not None:
        rating_prior = 0.5 if star_rating >= 4 else (-0.5 if star_rating <= 2 else 0.0)
    else:
        rating_prior = 0.0

    emoji_score = {"positive": 0.3, "negative": -0.3, "neutral": 0.0}.get(emoji_hint, 0.0)

    global_score, neg = 0.0, False
    for tok in stem_tokens:
        if tok in NEGATIONS: neg = True
        elif tok in POS_LEXICON: global_score += 1
        elif tok in NEG_LEXICON: global_score -= 1
    if neg: global_score = -global_score

    results = {}
    for asp, pos in aspect_positions:
        local = _score_window(stem_tokens, pos)
        if abs(local) < 0.05:
            norm = global_score / max(1, len(stem_tokens)) * 5
            combined = 0.4 * (rating_prior + emoji_score) + 0.6 * norm
        else:
            combined = 0.5 * local + 0.25 * rating_prior + 0.25 * emoji_score
        results[asp] = _score_to_label(combined)
    return results




class ABSAModel:
    """
    Two-head model:
      Head 1 → multi-label aspect detection   (TF-IDF char + word → OvR LogReg)
      Head 2 → per-aspect sentiment           (TF-IDF → LogReg per aspect)

    Falls back to rule-based when training data is scarce.
    """

    MIN_ASPECT_SAMPLES = 10   
    MIN_SENT_SAMPLES   = 8    
    def __init__(self):
        self.char_vec = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(2, 4),
            max_features=25000, sublinear_tf=True,
        )
        self.word_vec = TfidfVectorizer(
            analyzer="word", ngram_range=(1, 2),
            max_features=20000, sublinear_tf=True,
        )
        self.asp_mlb = MultiLabelBinarizer(classes=[a for a in ASPECTS if a != "none"])
        self.asp_clf: Optional[OneVsRestClassifier] = None
        self.sent_clfs: Dict[str, object] = {}
        self.asp_threshold = 0.25
        self.ml_ready = False

    def _feat(self, texts: List[str], fit: bool = False):
        if fit:
            Xc = self.char_vec.fit_transform(texts)
            Xw = self.word_vec.fit_transform(texts)
        else:
            Xc = self.char_vec.transform(texts)
            Xw = self.word_vec.transform(texts)
        return sp.hstack([Xc, Xw])

    def _preprocess_batch(self, df: pd.DataFrame) -> List[str]:
        texts = []
        for _, row in df.iterrows():
            r = preprocess(str(row["review_text"]), stem=True, remove_stops=True)
            # Enrich with metadata signals
            meta = ""
            if str(row.get("platform","")) in ("play_store","app_store"):
                meta = "تطبيق "
            texts.append(meta + r["clean_text"])
        return texts

    def fit(self, df: pd.DataFrame):
        texts = self._preprocess_batch(df)

        asp_labels = df["aspects"].tolist()
        # Filter out 'none' aspect
        asp_labels_clean = [[a for a in row if a in self.asp_mlb.classes]
                            for row in asp_labels]
        Y_asp = self.asp_mlb.fit_transform(asp_labels_clean)

        X = self._feat(texts, fit=True)
        self.asp_clf = OneVsRestClassifier(
            LogisticRegression(C=0.5, max_iter=500, class_weight="balanced",
                               solver="lbfgs")
        )
        self.asp_clf.fit(X, Y_asp)

        # Tune threshold on training set
        probs = self.asp_clf.predict_proba(X)
        # Find threshold that maximizes Jaccard on train
        best_j, best_t = 0.0, 0.25
        for t in np.arange(0.10, 0.60, 0.05):
            preds = [[self.asp_mlb.classes_[i] for i, p in enumerate(row) if p >= t]
                     for row in probs]
            preds = [p if p else ["general"] for p in preds]
            jac = np.mean([
                len(set(p) & set(t_)) / len(set(p) | set(t_))
                if (set(p) | set(t_)) else 1.0
                for p, t_ in zip(preds, asp_labels_clean)
            ])
            if jac > best_j:
                best_j, best_t = jac, t
        self.asp_threshold = best_t
        # print(f"[ML] Aspect threshold tuned to {self.asp_threshold:.2f} (train Jaccard={best_j:.3f})")

        # ── Sentiment heads ──
        # print("[ML] Training per-aspect sentiment classifiers …")
        for asp in ASPECTS[:-1]:
            mask = df["aspects"].apply(lambda a: asp in (a or []))
            n = mask.sum()
            if n < self.MIN_SENT_SAMPLES:
                # print(f"     {asp:20s}: skipped (only {n} samples) → rule-based fallback")
                continue
            sub_texts  = [texts[i] for i in df.index[mask]]
            sub_labels = df.loc[mask, "aspect_sentiments"].apply(
                lambda d: d.get(asp, "neutral") if isinstance(d, dict) else "neutral"
            ).tolist()
            if len(set(sub_labels)) < 2:
                self.sent_clfs[asp] = sub_labels[0]
                continue
            Xs = self.char_vec.transform(sub_texts)
            clf = LogisticRegression(C=0.7, max_iter=500, class_weight="balanced",
                                     solver="lbfgs")
            clf.fit(Xs, sub_labels)
            self.sent_clfs[asp] = clf
            print(f"     {asp:20s}: trained on {n} samples | classes={sorted(set(sub_labels))}")

        self.ml_ready = True
        # print("[ML] Training complete.\n")

    def predict_one(self, text: str, star_rating=None,
                    business_category: str = "", platform: str = "") -> dict:
        if not text or not text.strip():
            return {"aspects": ["none"], "aspect_sentiments": {"none": "neutral"}}

        proc  = preprocess(text, stem=True, remove_stops=True)
        clean = proc["clean_text"]
        hint  = proc["emoji_hint"]

        raw_proc  = preprocess(text, stem=False, remove_stops=False)
        stem_proc = preprocess(text, stem=True,  remove_stops=False)

        meta = "تطبيق " if platform in ("play_store","app_store") else ""
        enriched = meta + clean

        # Aspect detection
        detected = []
        if self.ml_ready and clean.strip():
            X = self._feat([enriched])
            probs = self.asp_clf.predict_proba(X)[0]
            detected = [self.asp_mlb.classes_[i]
                        for i, p in enumerate(probs)
                        if p >= self.asp_threshold]

        # Rule-based fallback
        if not detected:
            ap = rule_detect_aspects(raw_proc["tokens"], stem_proc["tokens"], platform)
            detected = [a for a, _ in ap]
            if not detected:
                detected = ["general"]

        # Sentiment per aspect
        sentiments: Dict[str, str] = {}
        for asp in detected:
            clf = self.sent_clfs.get(asp)
            if clf is None:
                # Full rule-based
                ap = rule_detect_aspects(raw_proc["tokens"], stem_proc["tokens"], platform)
                s  = rule_classify_sentiment(stem_proc["tokens"], ap, star_rating, hint)
                sentiments[asp] = s.get(asp, _score_to_label(
                    0.5 * (0.4 if hint=="positive" else -0.4 if hint=="negative" else 0.0) +
                    0.5 * (0.4 if (star_rating or 3) >= 4 else -0.4 if (star_rating or 3) <= 2 else 0.0)
                ))
            elif isinstance(clf, str):
                sentiments[asp] = clf
            else:
                Xs = self.char_vec.transform([enriched])
                sentiments[asp] = clf.predict(Xs)[0]

        # Validate keys
        sentiments = {k: v for k, v in sentiments.items() if k in detected}
        for asp in detected:
            if asp not in sentiments:
                sentiments[asp] = "neutral"

        return {"aspects": detected, "aspect_sentiments": sentiments}


# ───────────────────────────────────────────────────────────────────────
#  SECTION 5: DATA LOADING UTILITIES
# ───────────────────────────────────────────────────────────────────────

def load_df(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[-1].lower()
    if ext == ".xlsx":
        df = pd.read_excel(path, engine="openpyxl")
    elif ext == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")
    df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
    return df


def _parse_col(v):
    if isinstance(v, (list, dict)): return v
    try:
        _ = pd.isna(v)  # just a type check trigger
        if pd.isna(v):  return []
    except (TypeError, ValueError): pass
    try:   return json.loads(str(v).replace("'",'"'))
    except: return []


def prep_df(df: pd.DataFrame, labeled: bool = True) -> pd.DataFrame:
    for col in ["star_rating","business_category","platform","review_text"]:
        if col not in df.columns:
            df[col] = "" if col != "star_rating" else None
    def _star(v):
        try: return float(str(v).strip())
        except: return None
    df["star_rating"]       = df["star_rating"].apply(_star)
    df["review_text"]       = df["review_text"].fillna("").astype(str)
    df["business_category"] = df["business_category"].fillna("").astype(str)
    df["platform"]          = df["platform"].fillna("").astype(str)
    if labeled:
        if "aspects"           in df.columns: df["aspects"]           = df["aspects"].apply(_parse_col)
        if "aspect_sentiments" in df.columns: df["aspect_sentiments"] = df["aspect_sentiments"].apply(_parse_col)
    return df


# ───────────────────────────────────────────────────────────────────────
#  SECTION 6: PREPROCESSING DISPLAY
# ───────────────────────────────────────────────────────────────────────

def show_preprocessing(df: pd.DataFrame, n: int = 5):
    print("\n" + "="*72)
    print("  PREPROCESSING STEP-BY-STEP DISPLAY")
    print("="*72)

    STEP_LABELS = [
        ("original",            "Step 0 │ Original text"),
        ("after_emoji",         "Step 1 │ Emoji handling  → tags + hint"),
        ("after_franco",        "Step 2 │ Franco/Arabizi  → Arabic"),
        ("after_normalization",  "Step 3 │ Arabic normalization (alef, yeh, teh…)"),
        ("after_noise_removal",  "Step 4 │ Noise removal  (repeated chars)"),
        ("after_punctuation",    "Step 5 │ Punctuation removal"),
        ("after_numbers",        "Step 6 │ Number removal"),
        ("after_whitespace",     "Step 7 │ Whitespace normalisation"),
        ("tokens",               "Step 8 │ Tokenization"),
        ("after_stopwords",      "Step 9 │ Stop-word removal"),
        ("after_stemming",       "Step10 │ Light stemming"),
    ]

    samples = df.sample(min(n, len(df)), random_state=42)
    for _, row in samples.iterrows():
        print(f"\n{'─'*72}")
        print(f"  Review ID   : {row.get('review_id','N/A')}")
        print(f"  Platform    : {row.get('platform',''):<20}  Category: {row.get('business_category','')}")
        print(f"  Star rating : {row.get('star_rating','')}")

        r = preprocess(str(row["review_text"]), stem=True, remove_stops=True,
                       return_steps=True)
        s = r["steps"]
        for key, label in STEP_LABELS:
            print(f"\n  {label}")
            print(f"    {s.get(key,'')}")

        print(f"\n  ✅  Emoji hint   : {r['emoji_hint']}")
        print(f"  ✅  Final tokens : {r['tokens']}")
        print(f"  ✅  Clean text   : {r['clean_text']}")


# ───────────────────────────────────────────────────────────────────────
#  SECTION 7: EVALUATION
# ───────────────────────────────────────────────────────────────────────

def evaluate(model: ABSAModel, df: pd.DataFrame):
    print("\n" + "="*72)
    print("  train EVALUATION")
    print("="*72)

    preds_a, preds_s = [], []
    true_a,  true_s  = [], []

    for _, row in df.iterrows():
        p = model.predict_one(
            str(row["review_text"]),
            star_rating=row.get("star_rating"),
            business_category=str(row.get("business_category","")),
            platform=str(row.get("platform","")),
        )
        preds_a.append(set(p["aspects"]))
        preds_s.append(p["aspect_sentiments"])
        ta = row.get("aspects",[])
        true_a.append(set(ta) if isinstance(ta,list) else set())
        ts = row.get("aspect_sentiments",{})
        true_s.append(ts if isinstance(ts,dict) else {})

    # Exact match
    exact = sum(p == t for p, t in zip(preds_a, true_a)) / max(1, len(df))
    # Jaccard
    jac = np.mean([len(p&t)/len(p|t) if (p|t) else 1.0
                   for p, t in zip(preds_a, true_a)])
    # Sentiment accuracy on matched aspects
    c_s = t_s = 0
    for pa, ps, ta2, ts2 in zip(preds_a, preds_s, true_a, true_s):
        for asp in pa & ta2:
            t_s += 1
            if ps.get(asp) == ts2.get(asp): c_s += 1

    # print(f"\n  Aspect Exact-Match Accuracy   : {exact:.3f}")
    # print(f"  Aspect Mean Jaccard           : {jac:.3f}")
    sent_acc = c_s / max(1, t_s)
    print(f"  Sentiment Accuracy (matched)  : {sent_acc:.3f}  ({c_s}/{t_s})")

    print(f"\n  {'ID':>6}  {'True aspects':35s}  {'Pred aspects':35s}  Match")
    print(f"  {'─'*6}  {'─'*35}  {'─'*35}  ─────")
    for _, row in df.sample(min(12, len(df)), random_state=7).iterrows():
        p = model.predict_one(
            str(row["review_text"]),
            star_rating=row.get("star_rating"),
            platform=str(row.get("platform","")),
        )
        ta = sorted(row["aspects"]) if isinstance(row.get("aspects",[]),list) else []
        pa = sorted(p["aspects"])
        m  = "✅" if set(ta)==set(pa) else "❌"
        print(f"  {row['review_id']:>6}  {str(ta):35s}  {str(pa):35s}  {m}")
        print(f"         True sent: {row.get('aspect_sentiments',{})}")
        print(f"         Pred sent: {p['aspect_sentiments']}")


# ───────────────────────────────────────────────────────────────────────
#  SECTION 8: PREDICTION GENERATION
# ───────────────────────────────────────────────────────────────────────
VALID_ASPECTS = set([
    "food","service","price","cleanliness","delivery",
    "ambiance","app_experience","general","none"
])

def generate_predictions(model: ABSAModel, df: pd.DataFrame,
                        out_path: str = "predictions.json") -> List[dict]:

    print(f"\n[Predict] Processing {len(df)} reviews …")
    results = []

    for _, row in df.iterrows():
        p = model.predict_one(
            str(row["review_text"]),
            star_rating=row.get("star_rating"),
            business_category=str(row.get("business_category","")),
            platform=str(row.get("platform","")),
        )

        # ✅ Enforce valid aspects
        aspects = [a for a in p["aspects"] if a in VALID_ASPECTS]

        # ✅ If empty → fallback
        if not aspects:
            aspects = ["general"]

        # ✅ Remove duplicates
        aspects = list(set(aspects))

        # ✅ Clean sentiments
        sents = {}
        for asp in aspects:
            val = p["aspect_sentiments"].get(asp, "neutral")

            if val not in ["positive", "negative", "neutral"]:
                val = "neutral"

            sents[asp] = val

        results.append({
            "review_id": int(row["review_id"]),
            "aspects": aspects,
            "aspect_sentiments": sents,
        })

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"[Predict] ✅ Saved {len(results)} predictions → {out_path}")

    return results


    

In [33]:
train_path = r"F:\Uni a11\fourth\sec\123\DeepX_train.xlsx"
val_path = r"F:\Uni a11\fourth\sec\123\DeepX_validation.xlsx"
unlabeled_path = r"F:\Uni a11\fourth\sec\123\DeepX_hidden_test.xlsx"
output_path = r"F:\Uni a11\fourth\sec\123\predictions.json"

run(
    train_path=train_path,
    val_path=val_path,
    unlabeled_path=unlabeled_path,
    out_path=output_path,
    n_preview=5
)


  ARABIC ABSA PIPELINE  ──  DeepX Challenge

[Data] Loading: F:\Uni a11\fourth\sec\123\DeepX_train.xlsx
       1971 rows
[Data] Loading: F:\Uni a11\fourth\sec\123\DeepX_validation.xlsx
       500 rows
[Data] Loading: F:\Uni a11\fourth\sec\123\DeepX_hidden_test.xlsx
       500 rows

  PREPROCESSING STEP-BY-STEP DISPLAY

────────────────────────────────────────────────────────────────────────
  Review ID   : 873
  Platform    : google_maps           Category: مطعم مأكولات لبنانية
  Star rating : 5.0

  Step 0 │ Original text
    المكان جميل ومهدئ للاعصاب والاستاف كله جميل والاكل تحفة تحفة وحتي الاغاني بتخلي الموود وكل حاجة جميلة اوي ❤️❤️

  Step 1 │ Emoji handling  → tags + hint
    المكان جميل ومهدئ للاعصاب والاستاف كله جميل والاكل تحفة تحفة وحتي الاغاني بتخلي الموود وكل حاجة جميلة اوي  _EMOJI_ ️ _EMOJI_ ️

  Step 2 │ Franco/Arabizi  → Arabic
    المكان جميل ومهدئ للاعصاب والاستاف كله جميل والاكل تحفة تحفة وحتي الاغاني بتخلي الموود وكل حاجة جميلة اوي _EMOJI_ ️ _EMOJI_ ️

  Step 3 │ Arab

In [32]:
def evaluate_from_predictions(preds: List[dict], df: pd.DataFrame):
    print("  VALIDATION METRICS ")

    true_aspects = df["aspects"].tolist()
    true_sents   = df["aspect_sentiments"].tolist()

    pred_aspects = [set(p["aspects"]) for p in preds]
    true_aspects = [set(t) if isinstance(t, list) else set() for t in true_aspects]

    # ✅ Aspect Exact Match
    exact = sum(p == t for p, t in zip(pred_aspects, true_aspects)) / len(df)

    # ✅ Jaccard
    jac = sum(len(p & t)/len(p | t) if (p | t) else 1.0
              for p, t in zip(pred_aspects, true_aspects)) / len(df)

    # ✅ Sentiment accuracy
    correct = total = 0
    for p, t, ps, ts in zip(pred_aspects, true_aspects,
                            preds, true_sents):
        ts = ts if isinstance(ts, dict) else {}
        for asp in p & t:
            total += 1
            if ps["aspect_sentiments"].get(asp) == ts.get(asp):
                correct += 1

    sent_acc = correct / total if total else 0

    # print(f"Aspect Exact Match Accuracy   : {exact:.3f}")
    # print(f"Aspect Mean Jaccard           : {jac:.3f}")
    print(f"Sentiment Accuracy            : {sent_acc:.3f} ({correct}/{total})")

In [31]:
def run(train_path, val_path, unlabeled_path, out_path,
        n_preview=5, show_prep=True):

    print("  ARABIC ABSA PIPELINE  ──  DeepX Challenge")

    print(f"\n[Data] Loading: {train_path}")
    train_df = prep_df(load_df(train_path))
    print(f"       {len(train_df)} rows")

    print(f"[Data] Loading: {val_path}")
    val_df = prep_df(load_df(val_path))
    print(f"       {len(val_df)} rows")

    print(f"[Data] Loading: {unlabeled_path}")
    unlab_df = prep_df(load_df(unlabeled_path), labeled=False)
    print(f"       {len(unlab_df)} rows")

    if show_prep:
        show_preprocessing(train_df, n=n_preview)

    # ✅ Train ONLY on training data
    model = ABSAModel()
    model.fit(train_df)
    save_model(model, "absa_model.joblib")
    # ✅ Evaluate on TRAIN (to check overfitting)
    # print("  TRAIN EVALUATION")
    evaluate(model, train_df)

    val_input = val_df.drop(columns=["aspects", "aspect_sentiments"], errors="ignore")
    val_preds = generate_predictions(model, val_input, "val_predictions.json")

# Evaluate
    evaluate_from_predictions(val_preds, val_df)

    # ✅ Predict on unlabeled (test set)
    preds = generate_predictions(model, unlab_df, out_path)

    print(f" Done. Submission → {out_path}")

    # return preds
# Predict on validation instead of unlabeled


In [7]:
import joblib

def save_model(model: ABSAModel, path: str = "absa_model.joblib"):
    joblib.dump(model, path)
    print(f"[Model]  Saved model → {path}")


def load_model(path: str = "absa_model.joblib") -> ABSAModel:
    model = joblib.load(path)
    print(f"[Model]  Loaded model ← {path}")
    return model

In [73]:
import os
print("Saving model to:", os.path.abspath("absa_model.joblib"))

Saving model to: d:\visual\Microsoft VS Code\absa_model.joblib


In [9]:
import gradio as gr
import pandas as pd


def load_model():
    model = ABSAModel()

   
    return model


model = load_model()

# -----------------------------
# Prediction function
# -----------------------------
def predict_review(review, rating):
    if not review.strip():
        return pd.DataFrame([{"Aspect": "N/A", "Sentiment": "neutral"}])

    result = model.predict_one(
        text=review,
        star_rating=float(rating) if rating else None
    )

    aspects = result["aspects"]
    sentiments = result["aspect_sentiments"]

    rows = []
    for a in aspects:
        rows.append({
            "Aspect": a,
            "Sentiment": sentiments.get(a, "neutral")
        })

    return pd.DataFrame(rows)


# -----------------------------
# UI Design (Gradio Blocks)
# -----------------------------
with gr.Blocks(theme=gr.themes.Soft(), title="Arabic ABSA System") as app:

    gr.Markdown(
        """
        # 🇪🇬 Arabic Aspect-Based Sentiment Analysis (ABSA)
        ### DeepX Challenge Demo
        Enter a review → extract aspects + sentiment automatically
        """
    )

    with gr.Row():
        review_input = gr.Textbox(
            label="✍️ Enter Review",
            placeholder="اكتب تعليقك هنا... / Write your review here...",
            lines=5
        )

    rating_input = gr.Slider(
        minimum=1,
        maximum=5,
        step=1,
        label="⭐ Star Rating (optional)",
        value=3
    )

    btn = gr.Button("🔍 Analyze", variant="primary")

    output_table = gr.Dataframe(
        headers=["Aspect", "Sentiment"],
        label="📊 Results",
        interactive=False
    )

    btn.click(
        fn=predict_review,
        inputs=[review_input, rating_input],
        outputs=output_table
    )

    gr.Markdown(
        """
        ---
        ### 💡 Supported:
        - Arabic text 🇪🇬
        - Franco-Arabic (Arabizi)
        - Emojis 😊🔥💔
        - Restaurant / App reviews
        """
    )

# -----------------------------
# Run app
# -----------------------------
if __name__ == "__main__":
    app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
